# GPT-SoVITS V5 Emotion Control — Google Colab

이 노트북은 pymood1/GPT-SoVITS-improve의 현재 main 기준입니다.

- 기존 wav + .list를 다시 ASR/분할하지 않습니다.
- emotion_sidecar.jsonl이 없으면 기존 V5 전처리·backend·Gradio 테스트까지 수행하고 controller 학습만 건너뜁니다.
- 실제 평행 감정 발화만 pair_group_id로 묶어 학습합니다.
- controller는 S1과 V5 S2 SHA-256에 묶입니다.
- 감정 제어가 활성화된 요청은 현재 별도 가속 sampler를 우회합니다.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/pymood1/GPT-SoVITS-improve.git"
REPO_REF = "main"
REPO_DIR = Path("/content/GPT-SoVITS")

DRIVE_ROOT = Path("/content/drive/MyDrive")
LIST_PATH = DRIVE_ROOT / "audio_input/training.list"
WAV_ROOT = DRIVE_ROOT / "audio_input/datas"

SIDECAR_PATH = DRIVE_ROOT / "audio_input/emotion_sidecar.jsonl"
if not SIDECAR_PATH.exists():
    SIDECAR_PATH = None

WORKSPACE = DRIVE_ROOT / "audio_input/emotion_v5_workspace"
DATASET_ID = "my_voice"
DATASET_VERSION = "v1"
V5_VARIANT = "v5turbo"  # "v5dev" 또는 "v5turbo"
EMOTIONS = "neutral,happy,sad,angry"
CONTROLLER_STEPS = 500
GPU_ID = "0"

WORKSPACE.mkdir(parents=True, exist_ok=True)
print("LIST_PATH :", LIST_PATH)
print("WAV_ROOT  :", WAV_ROOT)
print("SIDECAR   :", SIDECAR_PATH)
print("WORKSPACE :", WORKSPACE)


In [ ]:
import os
import shutil
import subprocess

def run(cmd, cwd=None, env=None, check=True):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(cmd, cwd=cwd, env=env, check=check, text=True)

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    run(["git", "fetch", "--all", "--prune"], cwd=REPO_DIR)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(["git", "clone", REPO_URL, str(REPO_DIR)])

run(["git", "checkout", "--force", REPO_REF], cwd=REPO_DIR)
if REPO_REF == "main":
    run(["git", "reset", "--hard", "origin/main"], cwd=REPO_DIR)

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("[OK] checkout:", commit)
os.chdir(REPO_DIR)


In [ ]:
import shutil
import sys

if shutil.which("ffmpeg") is None:
    run(["apt-get", "update"])
    run(["apt-get", "install", "-y", "ffmpeg"])

run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"])
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO_DIR)
print("[OK] dependencies installed")


In [ ]:
PRETRAINED_URL = "https://huggingface.co/XXXXRT/GPT-SoVITS-Pretrained/resolve/main/pretrained_models.zip"
G2PW_URL = "https://huggingface.co/XXXXRT/GPT-SoVITS-Pretrained/resolve/main/G2PWModel.zip"

pretrained_dir = REPO_DIR / "GPT_SoVITS/pretrained_models"
g2pw_dir = REPO_DIR / "GPT_SoVITS/text/G2PWModel"

if not (pretrained_dir / "s1v3.ckpt").exists():
    zip_path = REPO_DIR / "pretrained_models.zip"
    run(["wget", "-q", "--show-progress", "-O", str(zip_path), PRETRAINED_URL])
    run(["unzip", "-q", "-o", str(zip_path), "-d", str(REPO_DIR / "GPT_SoVITS")])
    zip_path.unlink(missing_ok=True)

if not g2pw_dir.exists():
    zip_path = REPO_DIR / "G2PWModel.zip"
    run(["wget", "-q", "--show-progress", "-O", str(zip_path), G2PW_URL])
    run(["unzip", "-q", "-o", str(zip_path), "-d", str(REPO_DIR / "GPT_SoVITS/text")])
    zip_path.unlink(missing_ok=True)

print("[OK] pretrained models ready")


In [ ]:
import sys

run([sys.executable, "tools/check_emotion_patch.py"], cwd=REPO_DIR)

S1 = REPO_DIR / "GPT_SoVITS/pretrained_models/s1v3.ckpt"
S2 = REPO_DIR / f"GPT_SoVITS/pretrained_models/gsv-v5-pretrained/s2G{V5_VARIANT}.pth"
BERT = REPO_DIR / "GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large"
CNHUBERT = REPO_DIR / "GPT_SoVITS/pretrained_models/chinese-hubert-base"

missing = [str(p) for p in (S1, S2, BERT, CNHUBERT) if not p.exists()]
if missing:
    raise FileNotFoundError("필수 V5 파일이 없습니다:\n" + "\n".join(missing))

print("S1      :", S1)
print("S2      :", S2)
print("BERT    :", BERT)
print("CNHuBERT:", CNHUBERT)
print("[OK] emotion patch + V5 model files")


## 기존 데이터셋 import

training.list는 GPT-SoVITS 표준 wav_path|speaker|language|text 형식을 유지합니다. 기존 Colab 역슬래시 경로도 importer가 POSIX 경로로 정규화합니다.


In [ ]:
import json

if not LIST_PATH.is_file():
    raise FileNotFoundError(f"training.list를 찾을 수 없습니다: {LIST_PATH}")
if not WAV_ROOT.exists():
    raise FileNotFoundError(f"WAV_ROOT를 찾을 수 없습니다: {WAV_ROOT}")

lines = [x.rstrip("\r\n") for x in LIST_PATH.read_text(encoding="utf-8-sig").splitlines() if x.strip()]
print("list rows:", len(lines))
for line in lines[:5]:
    print(" ", line[:220])

if SIDECAR_PATH is None:
    template = WORKSPACE / "emotion_sidecar_template.jsonl"
    with template.open("w", encoding="utf-8") as f:
        for line in lines:
            wav = line.split("|", 1)[0]
            row = {
                "wav": wav,
                "source_id": None,
                "session_id": None,
                "split": "train",
                "emotion_intended": None,
                "pair_group_id": None,
                "pair_source": "none",
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print()
    print("[INFO] 감정 학습용 sidecar가 없습니다.")
    print("편집용 템플릿:", template)
    print("실제 같은 화자+같은 문장의 평행 녹음만 pair_group_id를 공유시키세요.")


In [ ]:
CANONICAL_DIR = WORKSPACE / f"canonical_{DATASET_VERSION}"
cmd = [
    sys.executable,
    "tools/emotion_dataset_builder.py",
    "--list", str(LIST_PATH),
    "--wav-root", str(WAV_ROOT),
    "--output", str(CANONICAL_DIR),
    "--dataset-id", DATASET_ID,
    "--dataset-version", DATASET_VERSION,
    "--stage-mode", "symlink",
]
if SIDECAR_PATH is not None:
    cmd += ["--sidecar", str(SIDECAR_PATH)]

run(cmd, cwd=REPO_DIR)

REPORT_PATH = CANONICAL_DIR / "report.json"
METADATA_PATH = CANONICAL_DIR / "metadata.jsonl"
TRAIN_LIST = CANONICAL_DIR / "train.list"
print("[OK] canonical dataset:", CANONICAL_DIR)


In [ ]:
import json
import soundfile as sf

report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
print(json.dumps(report, ensure_ascii=False, indent=2))

rows = [json.loads(line) for line in METADATA_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
train_rows = [row for row in rows if row.get("split") == "train"]
if not train_rows:
    raise RuntimeError("train split이 비어 있습니다.")

def ref_duration(row):
    path = CANONICAL_DIR / "wav" / row["canonical_wav"]
    try:
        info = sf.info(str(path))
        return float(info.frames) / float(info.samplerate)
    except Exception:
        return None

valid_ref_rows = []
for row in train_rows:
    duration = ref_duration(row)
    if duration is not None and 3.0 <= duration <= 10.0:
        item = dict(row)
        item["_duration"] = duration
        valid_ref_rows.append(item)

SMOKE_ROW = (
    min(valid_ref_rows, key=lambda row: abs(row["_duration"] - 5.0))
    if valid_ref_rows else None
)
if SMOKE_ROW is None:
    print("[WARNING] 3~10초 reference 후보가 없어 실제 /tts WAV smoke test는 생략됩니다.")
else:
    print("smoke reference:", SMOKE_ROW["canonical_wav"], f'{SMOKE_ROW["_duration"]:.2f}s')

pair_speakers = {
    row["speaker_uid"]
    for row in rows
    if row.get("split") == "train"
    and row.get("pair_group_id")
    and row.get("emotion_intended")
    and row.get("pair_source") in {"manual", "recording_manifest"}
}

reference_map = {}
for speaker_uid in sorted(pair_speakers):
    candidates = []
    for row in rows:
        if (
            row.get("speaker_uid") == speaker_uid
            and row.get("split") == "train"
            and not row.get("pair_group_id")
        ):
            duration = ref_duration(row)
            if duration is not None and 3.0 <= duration <= 10.0:
                item = dict(row)
                item["_duration"] = duration
                candidates.append(item)
    candidates.sort(key=lambda row: (
        0 if row.get("emotion_intended") == "neutral" else 1,
        0 if row.get("emotion_intended") in {None, "neutral"} else 1,
        abs(row["_duration"] - 5.0),
        row["canonical_wav"],
    ))
    if candidates:
        reference_map[speaker_uid] = candidates[0]["canonical_wav"]

REFERENCE_MAP_PATH = WORKSPACE / "reference_map.json"
REFERENCE_MAP_PATH.write_text(
    json.dumps(reference_map, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

pair_group_count = report.get("pair_groups_with_multiple_emotions", 0)
TRAINING_READY = bool(pair_group_count and pair_speakers and pair_speakers.issubset(reference_map))
print("pair speakers  :", sorted(pair_speakers))
print("reference map  :", reference_map)
print("training ready :", TRAINING_READY)


In [ ]:
FEATURE_ROOT = WORKSPACE / "features"
FEATURE_ROOT.mkdir(parents=True, exist_ok=True)

run([
    sys.executable,
    "tools/run_emotion_v5_preprocess.py",
    "--train-list", str(TRAIN_LIST),
    "--output-root", str(FEATURE_ROOT),
    "--bert-dir", str(BERT),
    "--cnhubert-dir", str(CNHUBERT),
    "--pretrained-s2g", str(S2),
    "--version", V5_VARIANT,
    "--gpu", GPU_ID,
], cwd=REPO_DIR)

matching = []
for manifest_path in FEATURE_ROOT.glob("build_*/build.json"):
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if (
        Path(manifest.get("train_list", "")).resolve() == TRAIN_LIST.resolve()
        and Path(manifest.get("pretrained_s2g", "")).resolve() == S2.resolve()
        and manifest.get("identity", {}).get("version") == V5_VARIANT
    ):
        matching.append(manifest_path)

if not matching:
    raise RuntimeError("현재 데이터/모델과 일치하는 preprocessing build를 찾지 못했습니다.")

BUILD_MANIFEST = max(matching, key=lambda p: p.stat().st_mtime)
FEATURE_DIR = BUILD_MANIFEST.parent
print("[OK] FEATURE_DIR:", FEATURE_DIR)


In [ ]:
import yaml

TTS_CONFIG = WORKSPACE / f"tts_{V5_VARIANT}.yaml"
payload = {
    "custom": {
        "device": "cuda",
        "is_half": True,
        "version": V5_VARIANT,
        "t2s_weights_path": str(S1),
        "vits_weights_path": str(S2),
        "bert_base_path": str(BERT),
        "cnhuhbert_base_path": str(CNHUBERT),
    }
}
TTS_CONFIG.write_text(
    yaml.safe_dump(payload, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)
print(TTS_CONFIG.read_text(encoding="utf-8"))


In [ ]:
REFERENCE_CACHE = WORKSPACE / f"reference_prompts_{V5_VARIANT}.pt"
CONTROLLER_PATH = WORKSPACE / f"emotion_controller_{V5_VARIANT}.pt"

if TRAINING_READY:
    run([
        sys.executable,
        "tools/export_emotion_reference_prompts.py",
        "--tts-config", str(TTS_CONFIG),
        "--metadata", str(METADATA_PATH),
        "--reference-map", str(REFERENCE_MAP_PATH),
        "--output", str(REFERENCE_CACHE),
    ], cwd=REPO_DIR)

    run([
        sys.executable,
        "tools/train_emotion_controller.py",
        "--s1-checkpoint", str(S1),
        "--feature-dir", str(FEATURE_DIR),
        "--feature-build-manifest", str(BUILD_MANIFEST),
        "--metadata", str(METADATA_PATH),
        "--reference-map", str(REFERENCE_MAP_PATH),
        "--reference-prompt-cache", str(REFERENCE_CACHE),
        "--output", str(CONTROLLER_PATH),
        "--emotions", EMOTIONS,
        "--steps", str(CONTROLLER_STEPS),
        "--device", "cuda",
    ], cwd=REPO_DIR)
    print("[OK] controller:", CONTROLLER_PATH)
else:
    print("[SKIP] pair metadata/reference가 충분하지 않아 controller 학습을 건너뜁니다.")


## API / Gradio smoke test

API는 9880, Gradio는 9872를 사용합니다. controller가 있으면 자동 로드합니다. sidecar가 없어도 첫 train 샘플로 실제 WAV 1회를 생성해 backend를 확인합니다.


In [ ]:
import requests
import time

API_LOG = WORKSPACE / "api_v2.log"
if "API_PROC" in globals() and API_PROC.poll() is None:
    API_PROC.terminate()
    API_PROC.wait(timeout=10)

api_env = os.environ.copy()
api_env["PYTHONUNBUFFERED"] = "1"
api_log_handle = API_LOG.open("w", encoding="utf-8")
API_PROC = subprocess.Popen(
    [sys.executable, "api_v2.py", "-a", "127.0.0.1", "-p", "9880", "-c", str(TTS_CONFIG)],
    cwd=REPO_DIR,
    env=api_env,
    stdout=api_log_handle,
    stderr=subprocess.STDOUT,
    text=True,
)

for _ in range(60):
    if API_PROC.poll() is not None:
        break
    try:
        r = requests.get("http://127.0.0.1:9880/docs", timeout=2)
        if r.status_code == 200:
            print("[OK] API docs reachable")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError("API가 제한 시간 안에 시작되지 않았습니다.")

if API_PROC.poll() is not None:
    print(API_LOG.read_text(encoding="utf-8", errors="replace")[-6000:])
    raise RuntimeError(f"api_v2.py exited: {API_PROC.returncode}")

if CONTROLLER_PATH.exists():
    r = requests.get(
        "http://127.0.0.1:9880/set_emotion_controller",
        params={"weights_path": str(CONTROLLER_PATH)},
        timeout=60,
    )
    print("controller load:", r.status_code, r.text[:1000])
    r.raise_for_status()


In [ ]:
if SMOKE_ROW is None:
    print("[SKIP] 3~10초 reference가 없어 실제 /tts WAV smoke test를 생략합니다.")
else:
    ref_row = SMOKE_ROW
    ref_audio = CANONICAL_DIR / "wav" / ref_row["canonical_wav"]
    body = {
        "text": ref_row["text"],
        "text_lang": ref_row["language"],
        "ref_audio_path": str(ref_audio),
        "prompt_text": ref_row["text"],
        "prompt_lang": ref_row["language"],
        "text_split_method": "cut5",
        "batch_size": 1,
        "media_type": "wav",
        "streaming_mode": False,
        "gpt_streaming": False,
    }
    if CONTROLLER_PATH.exists():
        body.update({"emotion_id": 0, "emotion_strength": 0.35})

    response = requests.post("http://127.0.0.1:9880/tts", json=body, timeout=180)
    print("status:", response.status_code, "bytes:", len(response.content))
    if response.status_code != 200:
        print(response.text[:3000])
        response.raise_for_status()

    BACKEND_SMOKE_WAV = WORKSPACE / "backend_smoke.wav"
    BACKEND_SMOKE_WAV.write_bytes(response.content)
    print("[OK] backend smoke wav:", BACKEND_SMOKE_WAV)


In [ ]:
GRADIO_LOG = WORKSPACE / "gradio.log"
LAUNCHER = WORKSPACE / "launch_gradio_emotion.py"
controller_literal = repr(str(CONTROLLER_PATH)) if CONTROLLER_PATH.exists() else "None"

launcher_lines = [
    "import sys",
    "from pathlib import Path",
    f"ROOT = Path({str(REPO_DIR)!r})",
    'sys.path.insert(0, str(ROOT / "GPT_SoVITS"))',
    "import inference_webui as webui",
    f"controller = {controller_literal}",
    "if controller:",
    "    webui.load_emotion_controller_webui(controller)",
    "webui.app.queue().launch(",
    '    server_name="0.0.0.0",',
    "    inbrowser=False,",
    "    share=True,",
    "    server_port=9872,",
    ")",
]
LAUNCHER.write_text("\n".join(launcher_lines) + "\n", encoding="utf-8")

if "GRADIO_PROC" in globals() and GRADIO_PROC.poll() is None:
    GRADIO_PROC.terminate()
    GRADIO_PROC.wait(timeout=10)

env = os.environ.copy()
env.update({
    "version": V5_VARIANT,
    "gpt_path": str(S1),
    "sovits_path": str(S2),
    "bert_path": str(BERT),
    "cnhubert_base_path": str(CNHUBERT),
    "is_half": "True",
    "is_share": "True",
    "infer_ttswebui": "9872",
    "PYTHONUNBUFFERED": "1",
})
log_handle = GRADIO_LOG.open("w", encoding="utf-8")
GRADIO_PROC = subprocess.Popen(
    [sys.executable, str(LAUNCHER)],
    cwd=REPO_DIR,
    env=env,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(15)
print(GRADIO_LOG.read_text(encoding="utf-8", errors="replace")[-8000:])
if GRADIO_PROC.poll() is not None:
    raise RuntimeError(f"Gradio process exited: {GRADIO_PROC.returncode}")
print("[OK] Gradio process running")


In [ ]:
for name in ("API_PROC", "GRADIO_PROC"):
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except Exception:
            proc.kill()
print("[OK] background processes stopped")
